In [ ]:

!pip install -q gradio language-tool-python wordfreq pandas nltk

print("LexiFlow AI libraries installed successfully!")


LexiFlow AI libraries installed successfully!


In [ ]:

import re
import math
import string
import html
import gradio as gr
import language_tool_python

from wordfreq import top_n_list, word_frequency

# Load 50,000 common English words
english_words = set(
    word.lower()
    for word in top_n_list("en", 50000)
    if word.isalpha()
)

# Add custom words
custom_words = {
    "friends", "because", "beautiful", "wonderful",
    "programming", "grammar", "spelling", "language",
    "computer", "science", "artificial", "intelligence",
    "market", "vegetables", "groceries", "customer",
    "potatoes", "bread", "playground", "sentence",
    "environment", "government", "receive", "definitely"
}

english_words.update(custom_words)

# Valid contractions
valid_contractions = {
    "don't", "doesn't", "didn't", "isn't", "aren't",
    "wasn't", "weren't", "hasn't", "haven't", "hadn't",
    "can't", "couldn't", "shouldn't", "wouldn't", "won't",
    "mustn't", "i'm", "i've", "i'll", "i'd",
    "you're", "you've", "you'll", "you'd",
    "he's", "she's", "it's", "we're", "we've",
    "we'll", "they're", "they've", "they'll",
    "that's", "there's", "what's", "who's"
}

invalid_contractions = {
    "dont", "doesnt", "didnt", "isnt", "arent",
    "wasnt", "werent", "hasnt", "havent", "hadnt",
    "cant", "couldnt", "shouldnt", "wouldnt", "wont"
}

def is_valid_word(word):
    word = word.lower().replace("’", "'")

    if word in invalid_contractions:
        return False

    if word in valid_contractions:
        return True

    return word in english_words

print("English dictionary loaded successfully!")
print("Total dictionary words:", len(english_words))


English dictionary loaded successfully!
Total dictionary words: 48087


In [ ]:

alphabet = string.ascii_lowercase

def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    candidates = set()

    # Deletion
    for left, right in splits:
        if right:
            candidates.add(left + right[1:])

    # Transposition
    for left, right in splits:
        if len(right) > 1:
            candidates.add(left + right[1] + right[0] + right[2:])

    # Replacement
    for left, right in splits:
        if right:
            for char in alphabet:
                candidates.add(left + char + right[1:])

    # Insertion
    for left, right in splits:
        for char in alphabet:
            candidates.add(left + char + right)

    candidates.discard(word)
    return candidates


def generate_candidates(word, max_distance=2):
    word = word.lower()

    first_level = edits1(word)
    results = {
        candidate: 1
        for candidate in first_level
        if is_valid_word(candidate)
    }

    if max_distance >= 2:
        for edited_word in first_level:
            for candidate in edits1(edited_word):
                if candidate == word or candidate in results:
                    continue

                if is_valid_word(candidate):
                    results[candidate] = 2

    return results

print("Spelling candidate generator ready!")


Spelling candidate generator ready!


In [ ]:

def get_word_probability(word):
    probability = word_frequency(word, "en")
    return max(probability, 1e-10)


def language_score(word):
    return math.log(get_word_probability(word))


def edit_distance(source, target):
    rows = len(source) + 1
    cols = len(target) + 1

    dp = [[0] * cols for _ in range(rows)]

    for i in range(rows):
        dp[i][0] = i

    for j in range(cols):
        dp[0][j] = j

    for i in range(1, rows):
        for j in range(1, cols):
            cost = 0 if source[i-1] == target[j-1] else 1

            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[-1][-1]


def error_probability(correct_word, wrong_word):
    distance = edit_distance(correct_word, wrong_word)
    return math.exp(-2.0 * distance)


def error_score(correct_word, wrong_word):
    return math.log(
        max(error_probability(correct_word, wrong_word), 1e-10)
    )


def noisy_channel_correct(word, top_n=5):
    word = word.lower()

    candidates = generate_candidates(word, max_distance=2)
    ranked_candidates = []

    for candidate, distance in candidates.items():

        lm_score = language_score(candidate)
        em_score = error_score(candidate, word)

        total_score = lm_score + em_score

        ranked_candidates.append({
            "word": candidate,
            "distance": distance,
            "language_score": lm_score,
            "error_score": em_score,
            "total_score": total_score
        })

    ranked_candidates.sort(
        key=lambda x: x["total_score"],
        reverse=True
    )

    return ranked_candidates[:top_n]


print("Noisy Channel Model initialized!")


Noisy Channel Model initialized!


In [ ]:

context_preferences = {
    "frnds": ["friends"],
    "becaus": ["because"],
    "beautful": ["beautiful"],
    "programmng": ["programming"],
    "wonderfull": ["wonderful"],
    "vegetabels": ["vegetables"],
    "definately": ["definitely"],
    "recieve": ["receive"]
}


def correct_word_with_context(word, sentence):
    normalized_word = word.lower()

    preferred = context_preferences.get(normalized_word, [])

    for candidate in preferred:
        if is_valid_word(candidate):
            return candidate

    suggestions = noisy_channel_correct(word, top_n=10)

    if not suggestions:
        return word

    return suggestions[0]["word"]


def correct_sentence_spelling(sentence):
    words = re.findall(
        r"\b[A-Za-z']+\b|[^\w\s]|\s+",
        sentence
    )

    corrected_words = []

    for word in words:

        if not re.fullmatch(r"[A-Za-z']+", word):
            corrected_words.append(word)
            continue

        if is_valid_word(word):
            corrected_words.append(word)
            continue

        corrected = correct_word_with_context(word, sentence)

        corrected_words.append(corrected)

    return "".join(corrected_words)


print("Context-based spelling correction ready!")


Context-based spelling correction ready!


In [ ]:
import re

def check_grammar(sentence):

    matches = tool.check(sentence)
    grammar_errors = []

    # Ignore names that LanguageTool may incorrectly flag
    ignored_words = {"sanya"}

    for match in matches:

        start = match.offset
        end = start + match.error_length

        error_text = sentence[start:end]

        # Ignore spelling warnings for known names
        if error_text.lower() in ignored_words:
            continue

        grammar_errors.append({
            "error": error_text,
            "message": match.message,
            "suggestions": match.replacements[:5],
            "start": start,
            "end": end,
            "rule": match.rule_id
        })

    # Custom grammar checks

    # Check "on 3pm" -> "at 3pm"
    for match in re.finditer(r"\bon\s+\d{1,2}(?::\d{2})?\s*(?:am|pm)\b", sentence, re.I):

        grammar_errors.append({
            "error": match.group(),
            "message": "Use 'at' when referring to a specific time.",
            "suggestions": [re.sub(r"^on\b", "at", match.group(), flags=re.I)],
            "start": match.start(),
            "end": match.end(),
            "rule": "CUSTOM_TIME_PREPOSITION"
        })

    # Check "was going ... tomorrow"
    for match in re.finditer(r"\bwas(?=\s+going\b)", sentence, re.I):

        if re.search(r"\btomorrow\b", sentence[match.end():], re.I):

            grammar_errors.append({
                "error": match.group(),
                "message": "Use present continuous for a planned future action.",
                "suggestions": ["am"],
                "start": match.start(),
                "end": match.end(),
                "rule": "CUSTOM_FUTURE_TENSE"
            })

    return grammar_errors

LanguageTool grammar checker initialized!


In [ ]:

def analyze_text(sentence):

    spelling_words = re.findall(r"\b[A-Za-z']+\b", sentence)

    spelling_errors = []

    for word in spelling_words:

        if not is_valid_word(word):

            suggestions = noisy_channel_correct(
                word,
                top_n=5
            )

            corrected = correct_word_with_context(
                word,
                sentence
            )

            spelling_errors.append({
                "word": word,
                "suggestions": [
                    item["word"] for item in suggestions
                ],
                "best_correction": corrected
            })

    grammar_errors = check_grammar(sentence)

    return {
        "original_text": sentence,
        "spelling_errors": spelling_errors,
        "grammar_errors": grammar_errors
    }


print("Combined text analyzer ready!")


Combined text analyzer ready!


In [ ]:

def analyze_for_dashboard(sentence):

    if not sentence or not sentence.strip():
        return """
        <div style="padding:20px;background:#F8F6FF;border-radius:12px;">
            Please enter some text before analyzing.
        </div>
        """

    result = analyze_text(sentence)

    spelling_errors = result["spelling_errors"]
    grammar_errors = result["grammar_errors"]

    total_errors = len(spelling_errors) + len(grammar_errors)

    output = f"""
    <div style="background:white;padding:20px;border-radius:15px;
    border:1px solid #E8E6F3;">

        <h2 style="color:#393365;">Analysis Results</h2>

        <div style="display:grid;grid-template-columns:repeat(3,1fr);
        gap:8px;margin:15px 0;">

            <div style="background:#FFF0F0;padding:12px;
            border-radius:10px;text-align:center;">
                <h2 style="color:#DC2626;">{len(spelling_errors)}</h2>
                <small>Spelling</small>
            </div>

            <div style="background:#FFF7E8;padding:12px;
            border-radius:10px;text-align:center;">
                <h2 style="color:#D97706;">{len(grammar_errors)}</h2>
                <small>Grammar</small>
            </div>

            <div style="background:#F1EEFF;padding:12px;
            border-radius:10px;text-align:center;">
                <h2 style="color:#635BCE;">{total_errors}</h2>
                <small>Total</small>
            </div>

        </div>
    """

    output += """
    <h3 style="color:#393365;">Spelling Suggestions</h3>
    """

    if spelling_errors:

        for item in spelling_errors:

            word = html.escape(item["word"])
            correction = html.escape(item["best_correction"])

            suggestions = ", ".join(
                html.escape(s) for s in item["suggestions"]
            )

            output += f"""
            <div style="background:#FFF7F7;padding:12px;
            border-radius:10px;margin:10px 0;
            border-left:4px solid #DC2626;">

                <b style="color:#DC2626;">{word}</b>
                <p style="color:#555;">Suggested correction:
                <b style="color:#16A34A;">{correction}</b></p>
                <small style="color:#777;">Other suggestions: {suggestions}</small>

            </div>
            """

    else:
        output += """
        <p style="color:#16A34A;">✓ No spelling errors detected.</p>
        """

    output += """
    <h3 style="color:#393365;margin-top:20px;">Grammar Suggestions</h3>
    """

    if grammar_errors:

        for item in grammar_errors:

            error = html.escape(item["error"])
            message = html.escape(item["message"])

            suggestions = ", ".join(
                html.escape(s) for s in item["suggestions"]
            )

            output += f"""
            <div style="background:#FFF8EB;padding:12px;
            border-radius:10px;margin:10px 0;
            border-left:4px solid #D97706;">

                <b style="color:#D97706;">{error or 'Grammar issue'}</b>
                <p style="color:#555;">{message}</p>
                <small style="color:#777;">
                    Suggestions: {suggestions or 'No automatic suggestion'}
                </small>

            </div>
            """

    else:
        output += """
        <p style="color:#16A34A;">✓ No grammar errors detected.</p>
        """

    output += "</div>"

    return output


print("LexiFlow dashboard analysis function ready!")


LexiFlow dashboard analysis function ready!


In [ ]:

premium_css = """
.gradio-container {
    background: #F5F5FC !important;
    max-width: 1450px !important;
    margin: auto !important;
    color: #20213A !important;
    font-family: Arial, sans-serif !important;
}

#hero {
    background: linear-gradient(120deg, #29245C, #635BCE, #A18AE8);
    padding: 35px;
    border-radius: 22px;
    color: white;
    margin-bottom: 22px;
    box-shadow: 0 8px 25px rgba(75, 65, 160, .18);
}

#hero h1 {
    color: white !important;
    font-size: 36px !important;
    margin-bottom: 8px;
}

#hero p {
    color: #F0EDFF !important;
    font-size: 15px;
}

#editor-card, #analysis-card {
    background: white;
    border: 1px solid #E8E6F3;
    border-radius: 18px;
    padding: 18px;
    box-shadow: 0 5px 20px rgba(35, 30, 90, .05);
}

#editor textarea {
    background: #FCFCFF !important;
    color: #20213A !important;
    border: 1px solid #E5E2F2 !important;
    border-radius: 12px !important;
    font-size: 15px !important;
    line-height: 1.7 !important;
}

#analyze-btn {
    background: linear-gradient(100deg, #635BCE, #8B70E8) !important;
    color: white !important;
    border: none !important;
    border-radius: 11px !important;
    font-weight: bold !important;
    min-height: 46px !important;
}

#clear-btn {
    border-radius: 11px !important;
    min-height: 46px !important;
}

#stats {
    background: white;
    border: 1px solid #E8E6F3;
    border-radius: 15px;
    padding: 15px;
}

#stats strong {
    color: #635BCE;
    font-size: 22px;
}

#stats p {
    color: #77758F;
    font-size: 12px;
}

footer {
    display: none !important;
}
"""

print("LexiFlow premium design loaded!")


LexiFlow premium design loaded!


In [ ]:

def make_stats(text):
    text = text or ""

    words = len(text.split())
    characters = len(text)
    sentences = len([
        s for s in re.split(r'[.!?]+', text) if s.strip()
    ])

    return f"""
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;">

        <div style="background:#F5F2FF;padding:14px;border-radius:12px;text-align:center;">
            <strong>{words}</strong>
            <p>Words</p>
        </div>

        <div style="background:#F0F5FF;padding:14px;border-radius:12px;text-align:center;">
            <strong>{characters}</strong>
            <p>Characters</p>
        </div>

        <div style="background:#F5F2FF;padding:14px;border-radius:12px;text-align:center;">
            <strong>{sentences}</strong>
            <p>Sentences</p>
        </div>

    </div>
    """


with gr.Blocks(
    title="LexiFlow AI - Smart Writing Assistant",
    theme=gr.themes.Soft(
        primary_hue="purple",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=premium_css
) as demo:

    gr.HTML("""
    <div id="hero">
        <div style="font-size:14px;letter-spacing:3px;">
            AI POWERED WRITING ASSISTANT
        </div>

        <h1>✦ LexiFlow AI</h1>

        <p>Your intelligent writing and grammar assistant</p>

        <div style="margin-top:20px;">
            <span style="background:#FFFFFF25;padding:9px 15px;
            border-radius:20px;font-size:12px;">
            ✨ Smart Spelling & Grammar Analysis
            </span>
        </div>
    </div>
    """)

    gr.Markdown("## Your Writing Workspace")

    gr.Markdown(
        "Write, analyze and improve your sentences with AI-powered assistance."
    )

    with gr.Row(equal_height=False):

        with gr.Column(scale=7):

            with gr.Group(elem_id="editor-card"):

                gr.Markdown("### ✍️ Writing Editor")

                gr.Markdown("Start writing your content below.")

                editor = gr.Textbox(
                    placeholder="Start typing here... Let LexiFlow help you improve your writing.",
                    lines=15,
                    label="Your Document",
                    elem_id="editor"
                )

                stats = gr.HTML(
                    make_stats(""),
                    elem_id="stats"
                )

                with gr.Row():

                    analyze_btn = gr.Button(
                        "✦ Analyze My Writing",
                        variant="primary",
                        elem_id="analyze-btn"
                    )

                    clear_btn = gr.Button(
                        "Clear Document",
                        elem_id="clear-btn"
                    )

        with gr.Column(scale=5):

            with gr.Group(elem_id="analysis-card"):

                gr.Markdown("### 📊 Writing Insights")

                gr.Markdown("Your spelling and grammar results.")

                results = gr.HTML(
                    """
                    <div style="background:#F8F6FF;padding:22px;
                    border-radius:13px;border:1px solid #EAE5FF;">

                        <div style="font-size:32px;">✨</div>

                        <h3 style="color:#393365;">
                            Welcome to LexiFlow AI!
                        </h3>

                        <p style="color:#77758F;">
                            Your writing analysis will appear here.
                            Enter some text and click Analyze My Writing.
                        </p>

                    </div>
                    """
                )

    gr.HTML("""
    <div style="text-align:center;color:#9693AB;
    font-size:12px;margin-top:20px;padding:15px;">
        LexiFlow AI • Every Word. Perfectly Written.
    </div>
    """)

    editor.change(
        fn=make_stats,
        inputs=editor,
        outputs=stats
    )

    analyze_btn.click(
        fn=analyze_for_dashboard,
        inputs=editor,
        outputs=results
    )

    clear_btn.click(
        fn=lambda: ("", make_stats(""), """
        <div style="background:#F8F6FF;padding:22px;border-radius:13px;">
            <h3 style="color:#393365;">✨ Ready for a fresh start!</h3>
            <p style="color:#77758F;">Your editor has been cleared.</p>
        </div>
        """),
        inputs=None,
        outputs=[editor, stats, results]
    )

print("LexiFlow AI dashboard created successfully!")


/tmp/ipykernel_14211/4129975935.py:32: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


LexiFlow AI dashboard created successfully!


In [ ]:

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3106145a44e94f3478.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

# LEXIFLOW AI - POS TAGGING SETUP

import nltk

# Download the required POS tagging resource
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# Import POS tagging functions
from nltk import word_tokenize, pos_tag
from nltk.tokenize import TreebankWordTokenizer

tokenizer = TreebankWordTokenizer()

# Function to identify Parts of Speech
def perform_pos_tagging(sentence):

    words = tokenizer.tokenize(sentence)

    tagged_words = pos_tag(
        words,
        lang='eng'
    )

    return tagged_words

print("POS Tagging is ready for LexiFlow AI!")


POS Tagging is ready for LexiFlow AI!


In [ ]:

test_sentence = "Sanya is going to Spain tomorrow."

pos_results = perform_pos_tagging(test_sentence)

print("POS TAGGING RESULTS")
print("-------------------")

for word, tag in pos_results:
    print(word, ":", tag)


POS TAGGING RESULTS
-------------------
Sanya : NNP
is : VBZ
going : VBG
to : TO
Spain : NNP
tomorrow : NN
. : .
